# \(B^\pm\to K^\pm K^\pm K^\mp\) — simultaneous direct-CP amplitude fit

This notebook targets the current **`main`** public API of `DalitzPlotFitter`.

Final convention:

- fit variables are fully **unfolded**;
- ROOT `s31` is mapped to internal `s13`;
- invariants are in GeV²;
- all resonances use pair `(0,2)`;
- `normalization_pair=(0,2)`;
- `normalize_components=True`;
- the standard full Square-Dalitz normalization grid is used unchanged;
- only the sideband background is folded in the ROOT file:
  `mPrime in [0,1]`, `thPrime in [0,0.5]`;
- background unfolding is done with
  `theta_fold = min(theta, 1-theta)`;
- prefit/postfit plots include `s13`, `s23`, `m13`, `m23`,
  1D SqDP projections (`mPrime`, `thetaPrime`), full 2D SqDP,
  and the unfolded Dalitz plane;
- Minuit defaults: verbosity 3, strategy 1.

Efficiency is built separately for B+ and B- from
`mclarge151617_kkk.root`, after applying the same data mass window.
For each charge the folded SqDP histogram is filled with
`weights=Event_PIDCalibEff` (sum of weights per bin), explicitly unfolded,
and only then smoothed.


Events with non-positive `Event_PIDCalibEff` are reported for diagnostics and excluded from the efficiency-map sample. The map itself is still filled with `weights=Event_PIDCalibEff`, i.e. the sum of weights in each SqDP bin.

Before filling the efficiency map, MC events with non-finite `mPrime`/`thPrime`, coordinates outside the folded SqDP domain, non-finite `Event_PIDCalibEff`, or `Event_PIDCalibEff <= 0` are reported and removed. The weighted histogram itself remains a plain sum of `Event_PIDCalibEff` per bin.

Implementation note: the notebook delegates fitting to `CPFitSession.fit`
(or `fit_multistart`) and delegates full SqDP histogram evaluation to
`SquareDalitzHistogramBackground` / `SquareDalitzHistogramEfficiency`.
Custom code is retained only to build the folded histograms from ROOT,
mirror them explicitly, smooth them in the requested order, and render
the extra prefit/postfit diagnostics not exposed by the public session API.


In [ ]:

from dataclasses import dataclass
from pathlib import Path
import inspect

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dalitzplotfitter import (
    BaBarFlatte,
    CPFitSession,
    CPRealImag,
    DalitzAmplitude,
    DecayChannel,
    DecayModel,
    Parameter,
    Resonance,
    SquareDalitzHistogramBackground,
    SquareDalitzHistogramEfficiency,
    enable_x64,
    invariants_to_square_dalitz,
    plot_binned_data,
    plot_dalitz,
    plot_square_dalitz,
    read_phase_space_sample,
    read_root_tree,
    weighted_resample,
)

enable_x64()

# Fail early if an older checkout is accidentally used.
_fit_signature = inspect.signature(CPFitSession.fit)
if 'strategy' not in _fit_signature.parameters or 'hesse' not in _fit_signature.parameters:
    raise RuntimeError(
        "This notebook requires the current DalitzPlotFitter main branch: "
        "CPFitSession.fit must expose strategy= and hesse=."
    )


## Configuration

In [ ]:
ROOT_FILE = Path('/home/jbaptist/work/data/B2KKK_data.root')
TREE_NAME = 'DecayTree'

MC_FILE = Path('/home/jbaptist/work/data/mclarge151617_kkk.root')
MC_TREE_NAME = 'DecayTree'

BRANCH = {
    'mass': 'B_m',
    'charge': 'B_charge',
    's12': 's12',
    's13': 's31',
    's23': 's23',
    'mprime': 'mPrime',
    'thetaprime': 'thPrime',
}

MC_BRANCH = {
    'mass': 'B_m',
    'charge': 'B_charge',
    'mprime': 'mPrime',
    'thetaprime': 'thPrime',
    'pid_eff': 'Event_PIDCalibEff',
}

INVARIANT_UNIT = 'GeV2'

BPLUS_CHARGE_VALUE = +1
BMINUS_CHARGE_VALUE = -1

SIGNAL_MASS_WINDOW = (5230.0, 5330.0)
BKG_MASS_MIN = 5500.0

SQDP_PAIR = (0, 2)
SQDP_BINS = (45, 45)
SQDP_SMOOTH_SIGMA = 1.0
SQDP_DENSITY_FLOOR = 1e-10
SEPARATE_BKG_BY_CHARGE = False

USE_EFFICIENCY = True
EFFICIENCY_BINS = (45, 45)
EFFICIENCY_SMOOTH_SIGMA = 1.0

SIGNAL_FRACTION_START = 0.90
FLOAT_SIGNAL_FRACTION = False

NORMALIZATION_RESOLUTION = 320

FIT_NSTARTS = 1
FIT_SEED = 20260903
FIT_TOLERANCE = 1e-4
FIT_VERBOSE = 3
FIT_STRATEGY = 1

# Phase-space pool used to build the discrete model CDF.
PROJECTION_POOL_SIZE = 1_000_000
# Oversample the unweighted toy to reduce MC fluctuations in projections.
PROJECTION_TOY_FACTOR = 5.0
PROJECTION_BINS = 60

## Load unfolded signal data and folded sideband SqDP

In [ ]:
def signal_cut(charge=None):
    lo, hi = SIGNAL_MASS_WINDOW
    cut = f"({BRANCH['mass']} >= {lo}) & ({BRANCH['mass']} <= {hi})"
    if charge is not None:
        cut += f" & ({BRANCH['charge']} == {charge})"
    return cut


def sideband_cut(charge=None):
    cut = f"({BRANCH['mass']} > {BKG_MASS_MIN})"
    if charge is not None:
        cut += f" & ({BRANCH['charge']} == {charge})"
    return cut


def convert_to_gev2(sample):
    if INVARIANT_UNIT == 'GeV2':
        return sample
    if INVARIANT_UNIT != 'MeV2':
        raise ValueError("INVARIANT_UNIT must be 'GeV2' or 'MeV2'")

    scale = 1e-6
    return PhaseSpaceSample(
        s12=jnp.asarray(sample.s12) * scale,
        s13=jnp.asarray(sample.s13) * scale,
        s23=jnp.asarray(sample.s23) * scale,
        weights=jnp.asarray(sample.weights),
        p1=sample.p1,
        p2=sample.p2,
        p3=sample.p3,
    )


def load_signal_sample(charge):
    sample = read_phase_space_sample(
        ROOT_FILE,
        TREE_NAME,
        s12=BRANCH['s12'],
        s13=BRANCH['s13'],
        s23=BRANCH['s23'],
        cut=signal_cut(charge),
    )
    return convert_to_gev2(sample)


def load_sideband_sqdp(charge=None):
    return read_root_tree(
        ROOT_FILE,
        TREE_NAME,
        {
            'mprime': BRANCH['mprime'],
            'thetaprime': BRANCH['thetaprime'],
        },
        cut=sideband_cut(charge),
    )


samples = {
    'plus': load_signal_sample(BPLUS_CHARGE_VALUE),
    'minus': load_signal_sample(BMINUS_CHARGE_VALUE),
}

bkg_sqdp = {
    'plus': load_sideband_sqdp(BPLUS_CHARGE_VALUE),
    'minus': load_sideband_sqdp(BMINUS_CHARGE_VALUE),
    'all': load_sideband_sqdp(),
}

raw_control = read_root_tree(
    ROOT_FILE,
    TREE_NAME,
    {'mass': BRANCH['mass'], 'charge': BRANCH['charge']},
)
raw_control = {name: np.asarray(values) for name, values in raw_control.items()}

print(f"B+ signal events: {samples['plus'].size:,}")
print(f"B- signal events: {samples['minus'].size:,}")
print(f"ROOT candidates:  {len(raw_control['mass']):,}")
def mc_signal_cut(charge):
    """Use exactly the data signal-mass window and one B charge."""
    lo, hi = SIGNAL_MASS_WINDOW
    return (
        f"({MC_BRANCH['mass']} >= {lo}) & "
        f"({MC_BRANCH['mass']} <= {hi}) & "
        f"({MC_BRANCH['charge']} == {charge})"
    )


def load_efficiency_mc(charge):
    """Load the folded SqDP coordinates and PID weight from MC."""
    return read_root_tree(
        MC_FILE,
        MC_TREE_NAME,
        {
            'mprime': MC_BRANCH['mprime'],
            'thetaprime': MC_BRANCH['thetaprime'],
            'pid_eff': MC_BRANCH['pid_eff'],
        },
        cut=mc_signal_cut(charge),
    )


def sanitize_efficiency_mc(coords, charge_label):
    """Remove entries that cannot contribute to a physical efficiency map."""
    mp = np.asarray(coords['mprime'], dtype=float)
    tp = np.asarray(coords['thetaprime'], dtype=float)
    pid_eff = np.asarray(coords['pid_eff'], dtype=float)

    finite_mp = np.isfinite(mp)
    finite_tp = np.isfinite(tp)
    finite_pid = np.isfinite(pid_eff)

    valid = (
        finite_mp
        & finite_tp
        & finite_pid
        & (mp >= 0.0)
        & (mp <= 1.0)
        & (tp >= 0.0)
        & (tp <= 0.5)
        & (pid_eff > 0.0)
    )

    n_total = int(mp.size)
    n_valid = int(np.count_nonzero(valid))
    n_removed = n_total - n_valid

    print(f"\nEfficiency MC sanitation: {charge_label}")
    print(f"  total in mass/charge selection : {n_total:,}")
    print(f"  valid for efficiency map       : {n_valid:,}")
    print(
        f"  removed                        : {n_removed:,} "
        f"({100.0*n_removed/max(n_total,1):.5f}%)"
    )
    print(f"    non-finite mPrime             : {np.count_nonzero(~finite_mp):,}")
    print(f"    non-finite thPrime            : {np.count_nonzero(~finite_tp):,}")
    print(f"    non-finite Event_PIDCalibEff  : {np.count_nonzero(~finite_pid):,}")
    print(
        "    finite PID efficiency <= 0    : "
        f"{np.count_nonzero(finite_pid & (pid_eff <= 0.0)):,}"
    )

    if n_valid == 0:
        raise RuntimeError(f"{charge_label}: no valid efficiency-MC events")

    return {
        'mprime': jnp.asarray(mp[valid]),
        'thetaprime': jnp.asarray(tp[valid]),
        'pid_eff': jnp.asarray(pid_eff[valid]),
    }


efficiency_mc_raw = {
    'plus': load_efficiency_mc(BPLUS_CHARGE_VALUE),
    'minus': load_efficiency_mc(BMINUS_CHARGE_VALUE),
}

efficiency_mc = {
    'plus': sanitize_efficiency_mc(efficiency_mc_raw['plus'], 'B+'),
    'minus': sanitize_efficiency_mc(efficiency_mc_raw['minus'], 'B-'),
}

print(f"\nEfficiency MC retained B+: {len(efficiency_mc['plus']['mprime']):,}")
print(f"Efficiency MC retained B-: {len(efficiency_mc['minus']['mprime']):,}")

In [ ]:

def validate_inputs():
    for charge, sample in samples.items():
        for variable in ('s12', 's13', 's23'):
            values = np.asarray(getattr(sample, variable), dtype=float)
            if np.any(~np.isfinite(values)):
                raise RuntimeError(f"{charge}: non-finite {variable}")

        s13 = np.asarray(sample.s13, dtype=float)
        s23 = np.asarray(sample.s23, dtype=float)
        print(
            f"{charge:5s}: "
            f"s13=[{s13.min():.4f},{s13.max():.4f}] GeV^2, "
            f"s23=[{s23.min():.4f},{s23.max():.4f}] GeV^2, "
            f"fraction(s13>s23)={np.mean(s13 > s23):.3f}"
        )

    for charge, coords in bkg_sqdp.items():
        mp = np.asarray(coords['mprime'], dtype=float)
        tp = np.asarray(coords['thetaprime'], dtype=float)

        if np.any(~np.isfinite(mp)) or np.any(~np.isfinite(tp)):
            raise RuntimeError(f"BKG {charge}: non-finite SqDP coordinates")
        if np.any((mp < 0.0) | (mp > 1.0)):
            raise RuntimeError(f"BKG {charge}: mPrime outside [0,1]")
        if np.any((tp < 0.0) | (tp > 0.5)):
            raise RuntimeError(f"BKG {charge}: thPrime outside [0,0.5]")

        print(
            f"BKG {charge:5s}: "
            f"mPrime=[{mp.min():.4f},{mp.max():.4f}], "
            f"thPrime=[{tp.min():.4f},{tp.max():.4f}]"
        )

    # Efficiency samples have already been sanitized; these are internal checks.
    for charge, coords in efficiency_mc.items():
        mp = np.asarray(coords['mprime'], dtype=float)
        tp = np.asarray(coords['thetaprime'], dtype=float)
        pid_eff = np.asarray(coords['pid_eff'], dtype=float)

        assert np.all(np.isfinite(mp))
        assert np.all(np.isfinite(tp))
        assert np.all(np.isfinite(pid_eff))
        assert np.all((0.0 <= mp) & (mp <= 1.0))
        assert np.all((0.0 <= tp) & (tp <= 0.5))
        assert np.all(pid_eff > 0.0)

        print(
            f"EFF {charge:5s}: N={len(mp):,}, "
            f"mPrime=[{mp.min():.4f},{mp.max():.4f}], "
            f"thPrime=[{tp.min():.4f},{tp.max():.4f}], "
            f"Event_PIDCalibEff=[{pid_eff.min():.6g},{pid_eff.max():.6g}]"
        )


validate_inputs()


## Square-Dalitz coordinates and background unfolding

In [ ]:

M_B_GEV = 5.27934
M_K_GEV = 0.493677
DAUGHTER_MASSES = (M_K_GEV, M_K_GEV, M_K_GEV)


def square_coordinates(data):
    return invariants_to_square_dalitz(
        jnp.asarray(data['s12']),
        jnp.asarray(data['s13']),
        jnp.asarray(data['s23']),
        mother_mass=M_B_GEV,
        masses=DAUGHTER_MASSES,
        pair=SQDP_PAIR,
    )


def smooth_histogram(values, sigma):
    values = np.asarray(values, dtype=float)
    if sigma is None or sigma <= 0:
        return values

    from scipy.ndimage import gaussian_filter
    return gaussian_filter(values, sigma=float(sigma), mode='nearest')


def mirror_folded_histogram(folded_hist):
    """Copy theta'=0->0.5 into theta'=1->0.5."""
    folded_hist = np.asarray(folded_hist, dtype=float)
    return np.concatenate(
        (folded_hist, folded_hist[:, ::-1]),
        axis=1,
    )


def build_background_map(coords):
    """Folded sideband histogram -> smoothing -> explicit full-theta mirror."""
    mp = np.asarray(coords['mprime'], dtype=float)
    tp = np.asarray(coords['thetaprime'], dtype=float)
    nx, ny_half = SQDP_BINS

    folded_hist, mp_edges, tp_edges_half = np.histogram2d(
        mp,
        tp,
        bins=(nx, ny_half),
        range=((0.0, 1.0), (0.0, 0.5)),
    )

    folded_hist = smooth_histogram(folded_hist, SQDP_SMOOTH_SIGMA)
    folded_hist = np.maximum(folded_hist, 0.0)

    if not np.any(folded_hist > 0.0):
        raise RuntimeError("Empty sideband histogram")

    full_hist = mirror_folded_histogram(folded_hist)
    tp_edges_full = np.linspace(0.0, 1.0, 2 * ny_half + 1)

    model = SquareDalitzHistogramBackground(
        mprime_edges=mp_edges,
        thetaprime_edges=tp_edges_full,
        values=full_hist,
        mother_mass=M_B_GEV,
        masses=DAUGHTER_MASSES,
        pair=SQDP_PAIR,
    )

    return (
        model,
        (folded_hist, mp_edges, tp_edges_half),
        (full_hist, mp_edges, tp_edges_full),
    )


def build_efficiency_map(coords):
    """PID-weighted folded MC -> explicit unfolding -> smoothing.

    The histogram is the SUM of Event_PIDCalibEff weights per bin.
    No mean-by-bin and no independent per-charge rescaling are applied.
    """
    mp = np.asarray(coords['mprime'], dtype=float)
    tp = np.asarray(coords['thetaprime'], dtype=float)
    pid_eff = np.asarray(coords['pid_eff'], dtype=float)
    nx, ny_half = EFFICIENCY_BINS

    folded_weighted, mp_edges, tp_edges_half = np.histogram2d(
        mp,
        tp,
        bins=(nx, ny_half),
        range=((0.0, 1.0), (0.0, 0.5)),
        weights=pid_eff,
    )
    folded_weighted = np.maximum(folded_weighted, 0.0)

    if not np.any(folded_weighted > 0.0):
        raise RuntimeError("Empty PID-weighted efficiency histogram")

    # User-requested order: unfolding first, smoothing second.
    unfolded_raw = mirror_folded_histogram(folded_weighted)
    tp_edges_full = np.linspace(0.0, 1.0, 2 * ny_half + 1)

    unfolded_smooth = smooth_histogram(
        unfolded_raw,
        EFFICIENCY_SMOOTH_SIGMA,
    )
    unfolded_smooth = np.maximum(unfolded_smooth, 0.0)

    model = SquareDalitzHistogramEfficiency(
        mprime_edges=mp_edges,
        thetaprime_edges=tp_edges_full,
        values=unfolded_smooth,
        mother_mass=M_B_GEV,
        masses=DAUGHTER_MASSES,
        pair=SQDP_PAIR,
    )

    return (
        model,
        (folded_weighted, mp_edges, tp_edges_half),
        (unfolded_raw, mp_edges, tp_edges_full),
        (unfolded_smooth, mp_edges, tp_edges_full),
    )


## BABAR Table-I model

In [ ]:
COEFFICIENT_STARTS = {
    # BaBar baseline starting values
    'phi1020': dict(rho=1.66, phase=2.99),
    'f0_980': dict(rho=5.20, phase=0.48),
    'X0_1550': dict(rho=8.20, phase=1.29),
    'f0_1710': dict(rho=1.22, phase=-0.59),
    'phi1680': dict(rho=0.10, phase=0.00),
    'chic0': dict(rho=0.437, phase=-1.02),
    'NR': dict(rho=13.2, phase=0.00),

    # Additional amplitudes: deliberately small neutral starting values.
    'f2_1270': dict(rho=0.10, phase=0.00),
    'f2p_1525': dict(rho=0.10, phase=0.00),
    'f0_1500': dict(rho=0.10, phase=0.00),
    'chic2': dict(rho=0.10, phase=0.00),
    'Jpsi_1S': dict(rho=0.10, phase=0.00),
}


def polar_to_xy(rho, phase):
    return float(rho * np.cos(phase)), float(rho * np.sin(phase))


def make_cp_coefficients():
    result = {}
    charmonium = {'chic0', 'chic2', 'Jpsi_1S'}

    for name, config in COEFFICIENT_STARTS.items():
        x0, y0 = polar_to_xy(config['rho'], config['phase'])
        is_reference = name == 'NR'
        is_charmonium = name in charmonium

        # Keep the existing NR global-reference convention.
        x = Parameter.coefficient(
            f'{name}.x',
            x0,
            owner=name,
            fixed=is_reference,
            step=0.01,
        )
        y = Parameter.coefficient(
            f'{name}.y',
            y0,
            owner=name,
            fixed=is_reference,
            step=0.01,
        )

        # Charmonium nominal convention: no intrinsic direct CPV.
        # NR keeps its existing convention: dx free, dy fixed.
        dx = Parameter.coefficient(
            f'{name}.dx',
            0.0,
            owner=name,
            fixed=is_charmonium,
            step=0.005,
        )
        dy = Parameter.coefficient(
            f'{name}.dy',
            0.0,
            owner=name,
            fixed=(is_reference or is_charmonium),
            step=0.005,
        )

        result[name] = CPRealImag(x, y, dx, dy)

    return result


@dataclass(frozen=True)
class BaBarExponentialNR:
    alpha: object

    def __call__(self, data):
        alpha = jnp.asarray(self.alpha)
        return (
            jnp.exp(-alpha * jnp.asarray(data['s13']))
            + jnp.exp(-alpha * jnp.asarray(data['s23']))
        ) / jnp.sqrt(2.0)


def build_models():
    shared = make_cp_coefficients()

    x0_mass = Parameter.dynamics(
        'X0_1550.mass',
        1.539,
        owner='X0_1550',
        backend_name='pole_mass',
        bounds=(1.40, 1.70),
        step=0.005,
    )
    x0_width = Parameter.dynamics(
        'X0_1550.width',
        0.257,
        owner='X0_1550',
        backend_name='pole_width',
        bounds=(0.05, 0.60),
        step=0.005,
    )
    nr_alpha = Parameter.dynamics(
        'NR.alpha',
        0.152,
        owner='NR',
        bounds=(0.0, 1.0),
        step=0.002,
    )

    def components(charge):
        coefficient = {
            name: cp_coefficient.for_charge(charge)
            for name, cp_coefficient in shared.items()
        }

        return [
            Resonance(
                'phi1020',
                (0, 2),
                coefficient['phi1020'],
                mass=1.019461,
                width=0.004249,
                spin=1,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'f0_980',
                (0, 2),
                coefficient['f0_980'],
                lineshape=BaBarFlatte(),
                mass=0.965,
                width=0.0,
                spin=0,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'X0_1550',
                (0, 2),
                coefficient['X0_1550'],
                mass=x0_mass,
                width=x0_width,
                spin=0,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'f0_1710',
                (0, 2),
                coefficient['f0_1710'],
                mass=1.715,
                width=0.125,
                spin=0,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'phi1680',
                (0, 2),
                coefficient['phi1680'],
                mass=1.680,
                width=0.150,
                spin=1,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'f2_1270',
                (0, 2),
                coefficient['f2_1270'],
                mass=1.2754,
                width=0.1866,
                spin=2,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'f2p_1525',
                (0, 2),
                coefficient['f2p_1525'],
                mass=1.5173,
                width=0.0720,
                spin=2,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'f0_1500',
                (0, 2),
                coefficient['f0_1500'],
                mass=1.522,
                width=0.108,
                spin=0,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'chic0',
                (0, 2),
                coefficient['chic0'],
                mass=3.41475,
                width=0.0104,
                spin=0,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'chic2',
                (0, 2),
                coefficient['chic2'],
                mass=3.55617,
                width=0.00194,
                spin=2,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            Resonance(
                'Jpsi_1S',
                (0, 2),
                coefficient['Jpsi_1S'],
                mass=3.096900,
                width=0.0000926,
                spin=1,
                resonance_radius=4.0,
                parent_radius=0.0,
            ),
            DalitzAmplitude(
                'NR',
                dynamics=BaBarExponentialNR(nr_alpha),
                coefficient=coefficient['NR'],
            ),
        ]

    plus = DecayModel(
        DecayChannel('B+', ('K+', 'K+', 'K-')),
        components(+1),
        normalize_components=True,
        normalization_resolution=NORMALIZATION_RESOLUTION,
        normalization_method='square-dalitz',
        normalization_pair=SQDP_PAIR,
    )

    minus = DecayModel(
        DecayChannel('B-', ('K-', 'K-', 'K+')),
        components(-1),
        normalize_components=True,
        normalization_resolution=NORMALIZATION_RESOLUTION,
        normalization_method='square-dalitz',
        normalization_pair=SQDP_PAIR,
    )

    return plus, minus, shared


plus_model, minus_model, shared_coefficients = build_models()

print("normalization_pair =", plus_model.normalization_pair)
print("B+ normalization points =", plus_model.normalization_sample.size)
print("B- normalization points =", minus_model.normalization_sample.size)

### \(\phi(1680)\)

The model keeps the existing \(f_0(1710)\) amplitude and now also includes
\(\phi(1680)\to K^+K^-\) as a spin-1 Breit-Wigner in `pair=(0,2)`, with fixed
nominal mass and width

\[
m_{\phi(1680)}=1.680~\mathrm{GeV},\qquad
\Gamma_{\phi(1680)}=0.150~\mathrm{GeV}.
\]

Its CP-even and CP-odd coefficient parameters are free, like the other
charmless amplitudes. The initial coefficient magnitude is 0.10 with phase 0.

### Additional test amplitudes

The model now also includes

\[
f_2(1270),\quad f_2'(1525),\quad f_0(1500),\quad
\chi_{c2}(1P),\quad J/\psi(1S).
\]

All are added to the \(K^+K^-\) isobar channel with fixed nominal mass/width
and floating CP coefficients. Their starting coefficient magnitude is 0.10.

**Numerical note:** \(J/\psi(1S)\) and \(\chi_{c2}(1P)\) are extremely narrow.
Their physical natural widths are kept here, but their normalization should be
checked carefully against the integration resolution before interpreting fitted
fractions or likelihood improvements.

### CP convention for charmonium

For the nominal model,

\[
\chi_{c0},\qquad \chi_{c2}(1P),\qquad J/\psi(1S)
\]

have

\[
dx=dy=0
\]

fixed. Their CP-even \(x,y\) coefficients remain free. Local CP asymmetry in
those mass regions can still arise through interference with the charmless
amplitudes.

The existing NR reference convention is unchanged.

## Simultaneous fit session

In [ ]:

def build_session():
    if SEPARATE_BKG_BY_CHARGE:
        plus_background, _, _ = build_background_map(bkg_sqdp['plus'])
        minus_background, _, _ = build_background_map(bkg_sqdp['minus'])
    else:
        common_background, _, _ = build_background_map(bkg_sqdp['all'])
        plus_background = common_background
        minus_background = common_background

    if FLOAT_SIGNAL_FRACTION:
        signal_fraction = Parameter(
            'signal_fraction',
            SIGNAL_FRACTION_START,
            bounds=(0.05, 0.999),
            step=0.005,
        )
    else:
        signal_fraction = float(SIGNAL_FRACTION_START)

    fit_session = CPFitSession(
        plus_model,
        minus_model,
        samples['plus'],
        samples['minus'],
        signal_fraction=signal_fraction,
    )

    if USE_EFFICIENCY:
        plus_efficiency, _, _, _ = build_efficiency_map(efficiency_mc['plus'])
        minus_efficiency, _, _, _ = build_efficiency_map(efficiency_mc['minus'])
        fit_session = fit_session.with_efficiency(
            plus_efficiency,
            minus_efficiency,
        )

    fit_session = fit_session.with_background(
        'combinatorial',
        plus_background,
        minus_shape=minus_background,
    )

    return fit_session


session = build_session()

for parameter in session.parameters:
    print(
        f"{parameter.name:24s} "
        f"value={parameter.value: .6g} fixed={parameter.fixed}"
    )


## Input plots

In [ ]:

def plot_histogram_map(histogram, title, theta_max, colorbar_label):
    values, mp_edges, tp_edges = histogram

    fig, ax = plt.subplots(figsize=(7.0, 5.2))
    image = ax.pcolormesh(
        mp_edges,
        tp_edges,
        np.asarray(values).T,
        shading='auto',
    )
    fig.colorbar(image, ax=ax, label=colorbar_label)

    if theta_max == 1.0:
        ax.axhline(0.5, linestyle='--', linewidth=1.0)

    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, theta_max)
    ax.set_xlabel(r"$m'$")
    ax.set_ylabel(r"$\theta'$")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


def plot_efficiency_stages(charge_label, coords):
    _, folded_weighted, unfolded_raw, unfolded_smooth = build_efficiency_map(coords)

    plot_histogram_map(
        folded_weighted,
        f'{charge_label}: PID-weighted folded MC',
        0.5,
        r"$\sum Event\_PIDCalibEff$",
    )
    plot_histogram_map(
        unfolded_raw,
        f'{charge_label}: explicitly unfolded weighted MC',
        1.0,
        r"$\sum Event\_PIDCalibEff$",
    )
    plot_histogram_map(
        unfolded_smooth,
        f'{charge_label}: unfolded + smoothed efficiency',
        1.0,
        'weighted efficiency map',
    )


fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(raw_control['mass'], bins=120, histtype='step')
ax.axvspan(*SIGNAL_MASS_WINDOW, alpha=0.15, label='signal window')
ax.axvline(BKG_MASS_MIN, linestyle='--', label='sideband start')
ax.set_xlabel(r'$B_m$ [MeV]')
ax.set_ylabel('Candidates')
ax.legend()
plt.show()

for charge, label in (('plus', r'$B^+$'), ('minus', r'$B^-$')):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    plot_dalitz(
        samples[charge],
        x='s13',
        y='s23',
        bins=70,
        ax=ax,
        title=f'{label} unfolded Dalitz',
    )
    plt.show()

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    plot_square_dalitz(
        samples[charge],
        mother_mass=M_B_GEV,
        masses=DAUGHTER_MASSES,
        pair=SQDP_PAIR,
        bins=60,
        ax=ax,
        title=f'{label} full Square Dalitz',
    )
    plt.show()

if SEPARATE_BKG_BY_CHARGE:
    _, plus_folded, plus_unfolded = build_background_map(bkg_sqdp['plus'])
    _, minus_folded, minus_unfolded = build_background_map(bkg_sqdp['minus'])

    plot_histogram_map(plus_folded, r'$B^+$ folded sideband', 0.5, 'sideband')
    plot_histogram_map(plus_unfolded, r'$B^+$ mirrored sideband', 1.0, 'sideband')
    plot_histogram_map(minus_folded, r'$B^-$ folded sideband', 0.5, 'sideband')
    plot_histogram_map(minus_unfolded, r'$B^-$ mirrored sideband', 1.0, 'sideband')
else:
    _, common_folded, common_unfolded = build_background_map(bkg_sqdp['all'])
    plot_histogram_map(common_folded, 'Combined folded sideband', 0.5, 'sideband')
    plot_histogram_map(common_unfolded, 'Combined mirrored sideband', 1.0, 'sideband')

if USE_EFFICIENCY:
    plot_efficiency_stages('B+', efficiency_mc['plus'])
    plot_efficiency_stages('B-', efficiency_mc['minus'])


## Prefit and postfit helpers — inverse-CDF model toys

The fitted total PDF is first evaluated on a large phase-space pool. Its cumulative
discrete distribution is then inverted by `weighted_resample` to generate an
unweighted model toy. All 1D and 2D projections are filled from that toy; the
weighted pool itself is never histogrammed. The toy is oversampled and normalized
back to the observed charge yield to suppress Monte Carlo fluctuations.

In [ ]:
def default_values():
    return {
        parameter.name: float(parameter.value)
        for parameter in session.parameters
    }


def _total_projection_weights(components, pool_size, charge):
    """Combine the fitted signal/background projection densities."""
    total = np.zeros(pool_size, dtype=float)
    for _, component_sample, weights in components:
        if component_sample.size != pool_size:
            raise RuntimeError(
                f'{charge}: projection component and phase-space pool differ in size'
            )
        total += np.asarray(weights, dtype=float)

    finite = np.isfinite(total)
    if not np.all(finite):
        raise RuntimeError(f'{charge}: non-finite total projection weights')

    # Tiny negative round-off is harmless, but a genuinely negative PDF is not.
    tolerance = 1e-12 * max(float(np.max(np.abs(total))), 1.0)
    if np.min(total) < -tolerance:
        raise RuntimeError(
            f'{charge}: total projection density has negative weights '
            f'(minimum={np.min(total):.6g})'
        )

    total = np.clip(total, 0.0, None)
    if not np.any(total > 0.0):
        raise RuntimeError(f'{charge}: empty total projection density')
    return jnp.asarray(total)


def generate_projection_toys(
    values,
    pool_size=PROJECTION_POOL_SIZE,
    toy_factor=PROJECTION_TOY_FACTOR,
    seed=FIT_SEED,
):
    """Generate unweighted model toys by inverse-CDF resampling.

    The discrete CDF is built from the complete fitted PDF: coherent signal,
    efficiency, signal fraction and combinatorial background.  The returned
    toys, rather than the weighted phase-space pool, are used in every plot.
    """
    plus_pool = plus_model.generate_phase_space(pool_size, seed=seed)
    minus_pool = minus_model.generate_phase_space(pool_size, seed=seed + 1)

    plus_components, minus_components = session._projection_components_pair(
        values,
        plus_pool,
        minus_pool,
    )

    plus_weights = _total_projection_weights(
        plus_components, plus_pool.size, 'B+'
    )
    minus_weights = _total_projection_weights(
        minus_components, minus_pool.size, 'B-'
    )

    n_plus = int(np.ceil(float(toy_factor) * samples['plus'].size))
    n_minus = int(np.ceil(float(toy_factor) * samples['minus'].size))

    plus_toy = weighted_resample(
        jax.random.key(seed + 2),
        plus_pool,
        plus_weights,
        n_plus,
        replace=True,
    )
    minus_toy = weighted_resample(
        jax.random.key(seed + 3),
        minus_pool,
        minus_weights,
        n_minus,
        replace=True,
    )

    print(
        f'Projection toys: B+={plus_toy.size:,}, B-={minus_toy.size:,} '
        f'(pool={pool_size:,}, factor={toy_factor:g})'
    )
    return {'plus': plus_toy, 'minus': minus_toy}


def _model_histogram(values, edges, data_size):
    """Histogram an oversampled toy and normalize it to the data yield."""
    values = np.asarray(values, dtype=float)
    scale = float(data_size) / max(values.size, 1)
    return np.histogram(values, bins=edges)[0].astype(float) * scale


def plot_one_dimensional(
    toys,
    squared,
    title_prefix,
    bins=PROJECTION_BINS,
):
    for variable, suffix in (('s13', '13'), ('s23', '23')):
        xlabel = (
            rf'$s_{{{suffix}}}=m^2_{{{suffix}}}$ [GeV$^2$]'
            if squared else rf'$m_{{{suffix}}}$ [GeV]'
        )
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

        for ax, key, charge in (
            (axes[0], 'plus', 'B+'),
            (axes[1], 'minus', 'B-'),
        ):
            data_values = np.asarray(getattr(samples[key], variable), dtype=float)
            toy_values = np.asarray(getattr(toys[key], variable), dtype=float)
            if not squared:
                data_values = np.sqrt(np.clip(data_values, 0.0, None))
                toy_values = np.sqrt(np.clip(toy_values, 0.0, None))

            edges = np.linspace(data_values.min(), data_values.max(), bins + 1)
            plot_binned_data(data_values, bins=edges, ax=ax, label='data')
            model_hist = _model_histogram(toy_values, edges, data_values.size)
            ax.stairs(model_hist, edges, linewidth=2.0, label='model toy')
            ax.set_xlabel(xlabel)
            ax.set_ylabel('Candidates / bin')
            ax.set_title(f'{title_prefix}: {charge}')
            ax.legend(fontsize=8)

        plt.show()


def plot_sqdp_projections(toys, title_prefix, bins=PROJECTION_BINS):
    """Plot data against unweighted inverse-CDF model toys in SqDP."""
    coordinates = {}
    for key in ('plus', 'minus'):
        coordinates[key] = {
            'data': square_coordinates(samples[key].as_dict()),
            'toy': square_coordinates(toys[key].as_dict()),
        }

    for index, xlabel in ((0, r"$m'$"), (1, r"$\theta'$")):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
        for ax, key, charge in (
            (axes[0], 'plus', 'B+'),
            (axes[1], 'minus', 'B-'),
        ):
            data_values = np.asarray(coordinates[key]['data'][index], dtype=float)
            toy_values = np.asarray(coordinates[key]['toy'][index], dtype=float)
            edges = np.linspace(0.0, 1.0, bins + 1)
            plot_binned_data(data_values, bins=edges, ax=ax, label='data')
            model_hist = _model_histogram(toy_values, edges, data_values.size)
            ax.stairs(model_hist, edges, linewidth=2.0, label='model toy')
            ax.set_xlim(0.0, 1.0)
            ax.set_xlabel(xlabel)
            ax.set_ylabel('Candidates / bin')
            ax.set_title(f'{title_prefix}: {charge} {xlabel}')
            ax.legend(fontsize=8)
        plt.show()


def _plot_2d_comparison(
    data_x,
    data_y,
    toy_x,
    toy_y,
    bins,
    ranges,
    title,
    xlabel,
    ylabel,
):
    data_hist, x_edges, y_edges = np.histogram2d(
        data_x, data_y, bins=bins, range=ranges
    )
    toy_hist, _, _ = np.histogram2d(
        toy_x, toy_y, bins=(x_edges, y_edges)
    )
    model_hist = toy_hist * (data_x.size / max(toy_x.size, 1))
    pull = (data_hist - model_hist) / np.sqrt(np.maximum(model_hist, 1.0))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)
    image = axes[0].pcolormesh(x_edges, y_edges, data_hist.T, shading='auto')
    fig.colorbar(image, ax=axes[0], label='data')
    axes[0].set_title(f'{title}: data')
    image = axes[1].pcolormesh(x_edges, y_edges, model_hist.T, shading='auto')
    fig.colorbar(image, ax=axes[1], label='model toy')
    axes[1].set_title(f'{title}: model toy')
    image = axes[2].pcolormesh(
        x_edges, y_edges, pull.T, shading='auto', vmin=-5, vmax=5
    )
    fig.colorbar(image, ax=axes[2], label='pull')
    axes[2].set_title(f'{title}: pull')
    for ax in axes:
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
    return fig, axes


def plot_sqdp_comparison(toys, title_prefix, bins=(45, 45)):
    for key, charge in (('plus', 'B+'), ('minus', 'B-')):
        data_mp, data_tp = square_coordinates(samples[key].as_dict())
        toy_mp, toy_tp = square_coordinates(toys[key].as_dict())
        _, axes = _plot_2d_comparison(
            np.asarray(data_mp), np.asarray(data_tp),
            np.asarray(toy_mp), np.asarray(toy_tp),
            bins, ((0.0, 1.0), (0.0, 1.0)),
            f'{title_prefix} {charge}', r"$m'$", r"$\theta'$",
        )
        for ax in axes:
            ax.set_xlim(0.0, 1.0)
            ax.set_ylim(0.0, 1.0)
        plt.show()


def plot_dalitz_comparison(toys, title_prefix, bins=(55, 55)):
    for key, charge in (('plus', 'B+'), ('minus', 'B-')):
        data = samples[key]
        toy = toys[key]
        data_s13 = np.asarray(data.s13, dtype=float)
        data_s23 = np.asarray(data.s23, dtype=float)
        toy_s13 = np.asarray(toy.s13, dtype=float)
        toy_s23 = np.asarray(toy.s23, dtype=float)
        ranges = (
            (min(data_s13.min(), toy_s13.min()), max(data_s13.max(), toy_s13.max())),
            (min(data_s23.min(), toy_s23.min()), max(data_s23.max(), toy_s23.max())),
        )
        _plot_2d_comparison(
            data_s13, data_s23, toy_s13, toy_s23,
            bins, ranges, f'{title_prefix} {charge}',
            r'$s_{13}$ [GeV$^2$]', r'$s_{23}$ [GeV$^2$]',
        )
        plt.show()


def plot_all_diagnostics(values, title_prefix, seed):
    # Generate each charge once and reuse the same unweighted model toy in all plots.
    toys = generate_projection_toys(values, seed=seed)
    plot_one_dimensional(toys, squared=True, title_prefix=title_prefix)
    plot_sqdp_projections(toys, title_prefix=title_prefix)
    plot_sqdp_comparison(toys, title_prefix=title_prefix)
    plot_dalitz_comparison(toys, title_prefix=title_prefix)
    plot_one_dimensional(toys, squared=False, title_prefix=title_prefix)


## Prefit diagnostics

In [ ]:
prefit_values = default_values()
plot_all_diagnostics(
    prefit_values,
    title_prefix='Prefit',
    seed=FIT_SEED + 1000,
)

## Fit with Minuit strategy 1

In [ ]:

if FIT_NSTARTS == 1:
    result = session.fit(
        simplex=False,
        strategy=FIT_STRATEGY,
        hesse=True,
        tolerance=FIT_TOLERANCE,
        verbose=FIT_VERBOSE,
    )
    scan_results = (result,)
else:
    multistart = session.fit_multistart(
        n_starts=FIT_NSTARTS,
        seed=FIT_SEED,
        include_default=True,
        simplex=False,
        strategy=FIT_STRATEGY,
        tolerance=FIT_TOLERANCE,
        verbose=FIT_VERBOSE,
    )
    result = multistart.best
    scan_results = multistart.results

fit_report = session.report(
    result,
    include_fit_fractions=False,
    include_correlation=True,
)


## Fit fractions

In [ ]:
fit_fractions = session.print_fit_fractions(
    result,
    acceptance_weighted=USE_EFFICIENCY,
    include_interference=False,
    precision=4,
)

plus_ff = fit_fractions['plus']
minus_ff = fit_fractions['minus']

component_names = [
    name for name in plus_ff
    if name in minus_ff
]

ff_table = pd.DataFrame({
    'component': component_names,
    'FF_plus': [float(plus_ff[name]) for name in component_names],
    'FF_minus': [float(minus_ff[name]) for name in component_names],
})

ff_table['FF_mean'] = 0.5 * (
    ff_table['FF_plus'] + ff_table['FF_minus']
)

display(ff_table)

## CP coefficients

In [ ]:
def build_cp_table():
    values = session.result_values(result)
    rows = []

    for name in COEFFICIENT_STARTS:
        x = values[f'{name}.x']
        y = values[f'{name}.y']
        dx = values[f'{name}.dx']
        dy = values[f'{name}.dy']

        c_plus = complex(x + dx, y + dy)
        c_minus = complex(x - dx, y - dy)

        mag_plus = abs(c_plus)
        mag_minus = abs(c_minus)

        phase_plus = np.angle(c_plus)
        phase_minus = np.angle(c_minus)

        denominator = mag_minus**2 + mag_plus**2

        acp = (
            (mag_minus**2 - mag_plus**2) / denominator
            if denominator > 0.0
            else np.nan
        )

        delta_phase = np.angle(
            np.exp(1j * (phase_minus - phase_plus))
        )

        rows.append({
            'component': name,
            '|c+|': mag_plus,
            '|c-|': mag_minus,
            'phi+ [rad]': phase_plus,
            'phi- [rad]': phase_minus,
            'A_CP(coeff)': acp,
            'delta_phi [rad]': delta_phase,
        })

    return pd.DataFrame(rows)


cp_table = build_cp_table()
display(cp_table)

## Postfit diagnostics

In [ ]:
postfit_values = session.result_values(result)

plot_all_diagnostics(
    postfit_values,
    title_prefix='Post-fit',
    seed=FIT_SEED + 5000,
)

## Sideband-background unfolding

The ROOT sideband is defined only on

\[
0\le m'\le1,\qquad 0\le\theta'\le0.5.
\]

The notebook now constructs the full background **explicitly** by reflection:

\[
\theta'_{\rm mirror}=1-\theta'.
\]

So the content from `theta'=0 -> 0.5` is copied to `theta'=1 -> 0.5`.
The resulting full histogram is normalized only after this mirroring.

The signal and normalization do not use the stored ROOT `mPrime/thPrime`.